In [ ]:
import pandas as pd
import datetime as dt

#  Let's quickly reload and clean or import our data ✌
df = pd.read_excel('../data/Online_Retail.xlsx', engine='openpyxl')
df_clean = df.dropna(subset=['CustomerID'])
df_final = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)].copy()
df_final['TotalSum'] = df_final['Quantity'] * df_final['UnitPrice']

# 2. Fix the date type to calculate time distances
df_final['InvoiceDate'] = pd.to_datetime(df_final['InvoiceDate'])

# 3. Establish a pinpoint evaluation date (1 day after the very last transaction in the history)
snapshot_date = df_final['InvoiceDate'].max() + dt.timedelta(days=1)

# 4. Group row records by CustomerID to map Recency, Frequency, and Monetary metrics
rfm_table = df_final.groupby(['CustomerID']).agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'count',                                   # Frequency
    'TotalSum': 'sum'                                       # Monetary
})

# 5. Rename for analytical clarity
rfm_table.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSum': 'Monetary'
}, inplace=True)

print("✅ RFM Aggregation Engine Complete!")
rfm_table.head()


In [ ]:
# 📊 Calculate key statistical boundaries for the new customer groups
print("--- Customer Base Overview ---")
print(f"Total Unique Customers Tracked: {rfm_table.shape[0]}\n")

# Review standard averages across the entire user base
print(rfm_table.describe())

<div style="background-color: #444C38; font-color: fff; padding: 15px; border-radius: 5px; font-size: 16px;">
    <p>
Looking at the Monetary row, the average (mean) customer spent about 💲2,054 but at least one top customer (max) spent an incredible 💲280,206!
    </p>
    </p>
Also, looking at Frequency: the 75th percentile is 100 orders, but the maximum is a massive 7,847 orders. This proves that the data has major outlier —heavy "whales" that skew the averages.
    </p>
</div>

In [ ]:
# 📊 Create quartile scoring functions
# Recency: Lower is better (1 = inactive for a long time, 4 = active recently)
r_labels = range(4, 0, -1) 
# Frequency & Monetary: Higher is better (1 = low engagement/spend, 4 = high engagement/spend)
f_labels = range(1, 5)
m_labels = range(1, 5)

# Cut the data into quartiles using pandas qcut
r_groups = pd.qcut(rfm_table['Recency'], q=4, labels=r_labels)
f_groups = pd.qcut(rfm_table['Frequency'], q=4, labels=f_labels)
m_groups = pd.qcut(rfm_table['Monetary'], q=4, labels=m_labels)

# Assign these new score columns back to our main table
rfm_table = rfm_table.assign(R=r_groups.values, F=f_groups.values, M=m_groups.values)

# Create an overall concatenated string score (e.g., 444, 111) and a combined numeric score
rfm_table['RFM_Segment'] = rfm_table['R'].astype(str) + rfm_table['F'].astype(str) + rfm_table['M'].astype(str)
rfm_table['RFM_Score'] = rfm_table[['R', 'F', 'M']].sum(axis=1)

print("✅ RFM Scores Assigned Successfully!")
rfm_table.head()


In [ ]:
# 🎯 Function to bucket numeric RFM scores into business segments
def segment_me(df):
    if df['RFM_Score'] >= 10:
        return '🥇 Champions / VIPs'
    elif (df['RFM_Score'] >= 7) and (df['RFM_Score'] < 10):
        return '🥈 Loyal & Active Customers'
    elif (df['RFM_Score'] >= 4) and (df['RFM_Score'] < 7):
        return '⚠️ At-Risk / Churn Risk'
    else:
        return '🛑 Lost Customers'

# Apply the segment names to your table
rfm_table['General_Segment'] = rfm_table.apply(segment_me, axis=1)

# Check the size of each newly formed group
print(rfm_table['General_Segment'].value_counts())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Count the frequency of each customer segment
segment_counts = rfm_table['General_Segment'].value_counts()

# 🧹 Clean the names for the chart axis to remove emoji warning errors
clean_index = [name.split(' ', 1)[1] if ' ' in name else name for name in segment_counts.index]

# Create a horizontal bar chart with text-only names
sns.barplot(
    x=segment_counts.values, 
    y=clean_index, 
    palette="viridis",
    hue=clean_index,
    legend=False
)

# Design polish
plt.title('E-Commerce Customer Segments Distribution', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Number of Customers', fontsize=12, labelpad=10)
plt.ylabel('', fontsize=12)

# Save the clean plot directly into your images folder for GitHub
plt.tight_layout()
plt.savefig('../images/customer_segments_distribution.png', dpi=300)
plt.show()
